# Formula.vLLM — reported vs reproduced

Reads `results/<pr>.json` (written by `record.py` → `derive.py`). Per PR, the **headline experiment** gives
`reported` = the PR author's speedup (baseline ÷ PR's solution, their machine) and `reproduced` = ours (this GPU),
with `ratio_sd` propagated from the two arms' run-to-run sd. Only real runs count (`summary.measured > 0`, `ts ≥ 2026-11-07`).

In [ ]:
import json, glob, os
import pandas as pd

HERE = os.path.dirname(os.path.abspath('stats.ipynb'))
rows = []
for f in sorted(glob.glob(os.path.join(HERE, 'results', '[0-9]*.json'))):
    d = json.load(open(f)); s = d.get('summary') or {}
    if not s.get('measured') or d.get('ts', '') < '2026-11-07':
        continue
    h = s.get('headline_derived') or {}; rp, rr = h.get('reproduced', {}), h.get('reported', {})
    rows.append(dict(pr=d['pr'], field=d.get('field'), headline=s.get('headline'), unit=h.get('unit'),
                     reported_baseline=rr.get('baseline'), reported_pr_solution=rr.get('pr_solution'), reported_ratio=rr.get('ratio'), reported_marker=rr.get('marker'),
                     reproduced_baseline=rp.get('baseline'), reproduced_pr_solution=rp.get('pr_solution'), reproduced_ratio=rp.get('ratio'), reproduced_ratio_sd=rp.get('ratio_sd'),
                     ratio_of_ratios=(h.get('reproduced_vs_reported') or {}).get('ratio_of_ratios'), verdict=s.get('verdict'),
                     reliability=(h.get('reliability') or {}).get('flag'), tests=f"{(s.get('tests') or {}).get('base')}/{(s.get('tests') or {}).get('gold')}",
                     title=d.get('title')))
df = pd.DataFrame(rows).sort_values('reproduced_ratio', ascending=False).reset_index(drop=True)
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 40)
print(f"{len(df)} PRs reproduced here · {int((df.verdict=='reproduced').sum())} match the reported ratio (≥0.8 of it)")
df[['pr','field','headline','unit','reported_baseline','reported_pr_solution','reported_ratio','reported_marker','reproduced_baseline','reproduced_pr_solution','reproduced_ratio','reproduced_ratio_sd','ratio_of_ratios','verdict','reliability','tests']]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

BLUE, ORANGE = '#0b8ce8', '#e39b00'   # reproduced / reported (same as the dashboard)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, max(3.5, 0.42 * len(df) + 1.5)), gridspec_kw={'width_ratios': [3, 2]})

# --- global: per PR, reproduced ratio ± sd vs reported ratio, log x ---
y = np.arange(len(df))[::-1]
ax1.errorbar(df.reproduced_ratio, y, xerr=df.reproduced_ratio_sd.fillna(0), fmt='o', color=BLUE, ecolor=BLUE, elinewidth=2, capsize=3, ms=7, label='reproduced ± sd (this GPU)', zorder=3)
m = df.reported_ratio.notna()
ax1.scatter(df.reported_ratio[m], y[m.values], marker='D', s=48, color=ORANGE, edgecolor='white', linewidth=.8, label='reported (PR author)', zorder=4)
flag = df.reliability.eq('below_host_floor')
if flag.any():
    ax1.scatter(df.reproduced_ratio[flag], y[flag.values], s=140, facecolors='none', edgecolors='#e5533f', linestyle='--', label='below host floor (ratio unreliable)', zorder=5)
ax1.axvline(1, color='#c9d6ec', lw=1.2, zorder=1)
ax1.set_xscale('log', base=2); ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}×")); ax1.set_yticks(y); ax1.set_yticklabels([f"#{p}  {h[:22]}" for p, h in zip(df.pr, df.headline.fillna(''))], fontsize=9)
ax1.set_xlabel('speedup = baseline ÷ PR\'s solution  (log2, 1× = no change)'); ax1.set_title('global: reported vs reproduced, per PR', loc='left', fontweight='bold')
ax1.grid(axis='x', color='#eef2f9'); ax1.spines[['top', 'right']].set_visible(False); ax1.legend(loc='lower right', fontsize=8, frameon=False)

# --- per field: reproduced ÷ reported (1 = exactly the author's gain), median with min–max range ---
g = df[df.ratio_of_ratios.notna()].groupby('field').ratio_of_ratios
pf = pd.DataFrame({'n': df.groupby('field').size(), 'k': g.size(), 'median': g.median(), 'lo': g.min(), 'hi': g.max()}).sort_values('n', ascending=False)
yy = np.arange(len(pf))[::-1]
ax2.barh(yy, (pf.hi - pf.lo).fillna(0), left=pf.lo.fillna(pf['median']), height=.45, color=BLUE, alpha=.25, label='min–max')
ax2.scatter(pf['median'], yy, color=BLUE, s=60, edgecolor='white', zorder=3, label='median')
for yi, (f, r) in zip(yy, pf.iterrows()):
    ax2.text((pf.hi.max() or 1) * 1.6, yi, f"n={int(r.n)}", va='center', fontsize=8, color='#5a6786')
ax2.axvline(1, color='#c9d6ec', lw=1.2); ax2.set_xscale('log', base=2); ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}")); ax2.set_xlim(right=(pf.hi.max() or 1) * 2.2); ax2.set_yticks(yy); ax2.set_yticklabels(pf.index, fontsize=9)
ax2.set_xlabel('reproduced ÷ reported  (1 = author\'s gain)'); ax2.set_title('per field', loc='left', fontweight='bold')
ax2.grid(axis='x', color='#eef2f9'); ax2.spines[['top', 'right']].set_visible(False); ax2.legend(loc='lower left', fontsize=8, frameon=False)
plt.tight_layout(); plt.show()
pf